In [13]:
import os
import configparser
from pyspark.sql import SparkSession

os.environ["LAB_BUCKET_NAME"] = os.environ["KUBERNETES_NAMESPACE"]
BUCKET = os.environ["LAB_BUCKET_NAME"]

# Lecture des identifiants du profil "default" (ceux qui fonctionnent)
cfg = configparser.ConfigParser()
cfg.read(os.path.expanduser("~/.aws/credentials"))
ACCESS_KEY = cfg["default"]["aws_access_key_id"]
SECRET_KEY = cfg["default"]["aws_secret_access_key"]
SESSION_TOKEN = cfg["default"].get("aws_session_token", "")

ICEBERG_PACKAGE = "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.11.0"

spark = (
    SparkSession.builder
    .appName("lab-silver-pyspark")
    .master("local[*]")
    # S3 access
    .config("spark.hadoop.fs.s3a.endpoint", f"https://{os.environ['AWS_S3_ENDPOINT']}")
    .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
    .config("spark.hadoop.fs.s3a.session.token", SESSION_TOKEN)
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider",
    )
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    # Iceberg
    .config("spark.jars.packages", ICEBERG_PACKAGE)
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.lab", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lab.type", "hadoop")
    .config("spark.sql.catalog.lab.warehouse", f"s3a://{BUCKET}/warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
spark

In [14]:
spark.sql("SHOW NAMESPACES IN lab").show()

+---------+
|namespace|
+---------+
|   silver|
+---------+



In [15]:
spark.sql("""
    SELECT count(*) AS orders,
           sum(quantity) AS quantity
    FROM lab.silver.orders
""").show()

+------+--------+
|orders|quantity|
+------+--------+
|  2833|    8628|
+------+--------+



In [16]:
spark.sql("""
    SELECT *
    FROM lab.silver.orders.snapshots
""").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                    |summary                 

In [17]:
spark.sql("""
    SELECT file_path,
           file_format,
           record_count,
           file_size_in_bytes
    FROM lab.silver.orders.files
""").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+
|file_path                                                                                                             |file_format|record_count|file_size_in_bytes|
+----------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+
|s3a://user-m-lebourdais-ece/warehouse/silver/orders/data/00000-6-684a2c74-8647-4b3b-af14-40f059fb51f7-0-00001.parquet |PARQUET    |3           |1953              |
|s3a://user-m-lebourdais-ece/warehouse/silver/orders/data/00000-57-e7ce37e1-778a-4df9-8374-11ad40472646-0-00001.parquet|PARQUET    |1           |1777              |
|s3a://user-m-lebourdais-ece/warehouse/silver/orders/data/00000-50-878a5d0e-79d4-4982-b386-11ebe94f027b-0-00001.parquet|PARQUET    |2829        |84055             |
+---------

In [18]:
from pyspark.sql import functions as F

stg_orders = spark.table("lab.silver.orders")
stg_orders.show(5)

+--------------------+--------------------+--------------------+--------+-------+
|            order_id|             user_id|          ordered_at|quantity|product|
+--------------------+--------------------+--------------------+--------+-------+
|3d5eb12c-3181-4fc...|f53d4330-cdda-44b...|2020-01-28 12:31:...|       2| cookie|
|11a11529-a807-4d6...|5f733a3e-5d28-4aa...|2020-04-18 22:51:...|       5|  bread|
|4e4c691a-aa85-416...|90164161-cfa7-41c...|2020-02-24 01:03:...|       4| cookie|
|       new-order-001|f53d4330-cdda-44b...|2020-01-28 12:31:...|       2| cookie|
|00060d5a-02b3-494...|f53d4330-cdda-44b...|2020-01-28 12:31:...|       2| cookie|
+--------------------+--------------------+--------------------+--------+-------+
only showing top 5 rows


In [19]:
new_orders = stg_orders.limit(3).withColumn("order_id", F.expr("uuid()"))

new_orders.writeTo(
    "lab.silver.orders"
).append()

In [20]:
spark.sql("""
    SELECT snapshot_id, operation, committed_at
    FROM lab.silver.orders.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-------------------+---------+-----------------------+
|snapshot_id        |operation|committed_at           |
+-------------------+---------+-----------------------+
|4589807972261435191|overwrite|2026-11-03 21:56:36.702|
|501571823824054514 |append   |2026-11-03 22:12:53.85 |
|2574236072035205297|append   |2026-11-06 15:29:36.732|
|6302971868723562094|append   |2026-11-06 15:33:00.198|
+-------------------+---------+-----------------------+



In [21]:
snapshots = spark.sql("""
    SELECT snapshot_id
    FROM lab.silver.orders.snapshots
    ORDER BY committed_at
""").collect()

snapshots

[Row(snapshot_id=4589807972261435191),
 Row(snapshot_id=501571823824054514),
 Row(snapshot_id=2574236072035205297),
 Row(snapshot_id=6302971868723562094)]

In [22]:
first_snapshot = snapshots[0]["snapshot_id"]

spark.sql(f"""
    SELECT count(*)
    FROM lab.silver.orders
    VERSION AS OF {first_snapshot}
""").show()

+--------+
|count(1)|
+--------+
|    2829|
+--------+



In [23]:
spark.sql("""
    SELECT count(*)
    FROM lab.silver.orders
""").show()

+--------+
|count(1)|
+--------+
|    2836|
+--------+



In [24]:
spark.sql("""
    ALTER TABLE lab.silver.orders
    ADD COLUMN source STRING
""")

DataFrame[]

In [25]:
spark.sql("""
    DESCRIBE lab.silver.orders
""").show()

+----------+---------+-------+
|  col_name|data_type|comment|
+----------+---------+-------+
|  order_id|   string|   NULL|
|   user_id|   string|   NULL|
|ordered_at|timestamp|   NULL|
|  quantity|      int|   NULL|
|   product|   string|   NULL|
|    source|   string|   NULL|
+----------+---------+-------+



In [26]:
spark.sql("""
    INSERT INTO lab.silver.orders
    SELECT order_id, user_id, ordered_at, quantity, product, 'spark'
    FROM lab.silver.orders
    LIMIT 1
""")

DataFrame[]

In [27]:
spark.sql("""
    SELECT source, count(*) AS nb
    FROM lab.silver.orders
    GROUP BY source
""").show()

+------+----+
|source|  nb|
+------+----+
|  NULL|2836|
| spark|   1|
+------+----+



In [28]:
stg_users = spark.table("lab.silver.users")

stg_users.printSchema()
stg_users.show(5)

spark.sql("SELECT count(*) AS users FROM lab.silver.users").show()

spark.sql("""
    SELECT snapshot_id, operation, committed_at
    FROM lab.silver.users.snapshots
""").show(truncate=False)

root
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- name: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- email: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)

+--------------------+--------------+------------------+---+--------------------+----------+--------------------+--------------+-----+--------+
|             user_id|      username|              name|sex|               email| birthdate|              street|          city|state|zip_code|
+--------------------+--------------+------------------+---+--------------------+----------+--------------------+--------------+-----+--------+
|bdd640fb-0667-4ad...|  garzaanthony|    Charles Garcia|  M|helenpeterson@gma...|1935-10-04|908 Jennifer Squares| Robinsonshire|   KY|   01352|
|17be3111-1a2a-43e...|   blairamanda|    

In [29]:
from pyspark.sql import functions as F

stg_users = spark.table("lab.silver.users")

new_users = stg_users.limit(3).withColumn("user_id", F.expr("uuid()"))

new_users.writeTo("lab.silver.users").append()

In [30]:
spark.sql("""
    SELECT snapshot_id, operation, committed_at
    FROM lab.silver.users.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-------------------+---------+-----------------------+
|snapshot_id        |operation|committed_at           |
+-------------------+---------+-----------------------+
|8946536727875333288|overwrite|2026-11-03 21:56:34.295|
|5815796395399129591|append   |2026-11-06 15:37:26.689|
+-------------------+---------+-----------------------+



In [31]:
users_snapshots = spark.sql("""
    SELECT snapshot_id
    FROM lab.silver.users.snapshots
    ORDER BY committed_at
""").collect()

first_users_snapshot = users_snapshots[0]["snapshot_id"]

print("Avant l'ajout :")
spark.sql(f"""
    SELECT count(*) AS users
    FROM lab.silver.users
    VERSION AS OF {first_users_snapshot}
""").show()

print("Après l'ajout :")
spark.sql("""
    SELECT count(*) AS users
    FROM lab.silver.users
""").show()

Avant l'ajout :
+-----+
|users|
+-----+
|   50|
+-----+

Après l'ajout :
+-----+
|users|
+-----+
|   53|
+-----+



In [32]:
spark.sql("""
    ALTER TABLE lab.silver.users
    ADD COLUMN country STRING
""")

DataFrame[]

In [33]:
users_with_country = (
    spark.table("lab.silver.users")
    .limit(2)
    .withColumn("user_id", F.expr("uuid()"))
    .withColumn("country", F.lit("France"))
)

users_with_country.writeTo("lab.silver.users").append()

In [34]:
spark.sql("DESCRIBE lab.silver.users").show()

spark.sql("""
    SELECT country, count(*) AS nb
    FROM lab.silver.users
    GROUP BY country
""").show()

+---------+---------+-------+
| col_name|data_type|comment|
+---------+---------+-------+
|  user_id|   string|   NULL|
| username|   string|   NULL|
|     name|   string|   NULL|
|      sex|   string|   NULL|
|    email|   string|   NULL|
|birthdate|     date|   NULL|
|   street|   string|   NULL|
|     city|   string|   NULL|
|    state|   string|   NULL|
| zip_code|   string|   NULL|
|  country|   string|   NULL|
+---------+---------+-------+

+-------+---+
|country| nb|
+-------+---+
| France|  2|
|   NULL| 53|
+-------+---+



In [35]:
spark.stop()